# 🔍 Data Quality Framework

**Purpose.** Define reusable assertion functions that validate the data contracts at every layer of the medallion pipeline.

**Philosophy.** Each layer of the medallion pattern has implicit contracts. Bronze must contain every source row. Silver must contain only clean, typed data within sensible value ranges. Gold must aggregate to numbers that reconcile back to Silver. These functions make those contracts explicit and testable.

**Pattern.** Each check is a function returning `(passed: bool, message: str)`. The runner notebook (in `notebooks/qa/`) calls all of them and prints a pass/fail summary.

**Why custom assertions instead of PyDeequ/Great Expectations?** For a project this size, custom Spark assertions are simpler, have zero extra dependencies, and demonstrate the underlying pattern. In production we'd graduate to a proper framework once the assertion library grew past ~20 checks.

In [0]:
from typing import Tuple
from pyspark.sql import DataFrame
from pyspark.sql import functions as F

# Each check returns (passed, message). Standardising this makes the runner
# trivially loop over a list of (name, fn) pairs and produce a summary.

def _ok(message: str) -> Tuple[bool, str]:
    return True, message

def _fail(message: str) -> Tuple[bool, str]:
    return False, message

In [0]:
# Bronze layer contract:
#   - Table exists and is non-empty
#   - Audit columns are populated for every row
#   - Source-file column shows the expected files

def bronze_table_not_empty(spark) -> Tuple[bool, str]:
    n = spark.table("workspace.bronze.yellow_taxi").count()
    if n == 0:
        return _fail("Bronze table is empty.")
    return _ok(f"Bronze table has {n:,} rows.")

def bronze_audit_columns_populated(spark) -> Tuple[bool, str]:
    df = spark.table("workspace.bronze.yellow_taxi")
    null_ingested = df.filter(F.col("_ingested_at").isNull()).count()
    null_source = df.filter(F.col("_source_file").isNull()).count()
    if null_ingested > 0 or null_source > 0:
        return _fail(
            f"Bronze has null audit columns: "
            f"_ingested_at null={null_ingested:,}, _source_file null={null_source:,}."
        )
    return _ok("All Bronze rows have populated audit columns.")

def bronze_source_files_expected(spark) -> Tuple[bool, str]:
    df = spark.table("workspace.bronze.yellow_taxi")
    files = [r["_source_file"] for r in df.select("_source_file").distinct().collect()]
    expected_months = ["2024-01", "2024-02"]
    matched = [m for m in expected_months if any(m in f for f in files)]
    if len(matched) != len(expected_months):
        return _fail(f"Expected source files for {expected_months}; saw {files}.")
    return _ok(f"Bronze sourced from expected months: {expected_months}.")

In [0]:
# Silver layer contract:
#   - Row count is less than Bronze (we dropped bad rows) but within a sensible range
#   - All fares are positive
#   - All trip distances are positive
#   - Dropoff is always after pickup
#   - Pickup timestamps are inside the months we ingested
#   - Zone columns are populated (the join worked)

def silver_row_count_sensible(spark) -> Tuple[bool, str]:
    bronze = spark.table("workspace.bronze.yellow_taxi").count()
    silver = spark.table("workspace.silver.yellow_taxi").count()
    if silver == 0:
        return _fail("Silver table is empty.")
    if silver > bronze:
        return _fail(f"Silver ({silver:,}) > Bronze ({bronze:,}). Cleaning should drop rows, not add them.")
    drop_pct = (bronze - silver) / bronze * 100
    if drop_pct > 20:
        return _fail(f"Silver dropped {drop_pct:.1f}% of Bronze — that's higher than expected.")
    return _ok(f"Silver has {silver:,} rows; dropped {drop_pct:.2f}% of Bronze.")

def silver_no_negative_fares(spark) -> Tuple[bool, str]:
    n = spark.table("workspace.silver.yellow_taxi").filter(F.col("fare_amount") <= 0).count()
    if n > 0:
        return _fail(f"Silver has {n:,} rows with fare_amount <= 0.")
    return _ok("All Silver fares are positive.")

def silver_no_zero_distance(spark) -> Tuple[bool, str]:
    n = spark.table("workspace.silver.yellow_taxi").filter(F.col("trip_distance") <= 0).count()
    if n > 0:
        return _fail(f"Silver has {n:,} rows with trip_distance <= 0.")
    return _ok("All Silver trip distances are positive.")

def silver_dropoff_after_pickup(spark) -> Tuple[bool, str]:
    n = spark.table("workspace.silver.yellow_taxi").filter(
        F.col("tpep_dropoff_datetime") <= F.col("tpep_pickup_datetime")
    ).count()
    if n > 0:
        return _fail(f"Silver has {n:,} rows where dropoff <= pickup.")
    return _ok("All Silver dropoff timestamps follow their pickup timestamps.")

def silver_pickup_in_expected_window(spark) -> Tuple[bool, str]:
    df = spark.table("workspace.silver.yellow_taxi")
    n_before = df.filter(F.col("tpep_pickup_datetime") < "2024-01-01").count()
    n_after = df.filter(F.col("tpep_pickup_datetime") >= "2024-03-01").count()
    if n_before > 0 or n_after > 0:
        return _fail(f"Silver has rows outside [2024-01, 2024-03): before={n_before:,}, after={n_after:,}.")
    return _ok("All Silver pickups inside [2024-01-01, 2024-03-01).")

def silver_zones_populated(spark) -> Tuple[bool, str]:
    df = spark.table("workspace.silver.yellow_taxi")
    n_pickup = df.filter(F.col("pickup_zone").isNull()).count()
    n_dropoff = df.filter(F.col("dropoff_zone").isNull()).count()
    total = df.count()
    pickup_pct = n_pickup / total * 100
    dropoff_pct = n_dropoff / total * 100
    if pickup_pct > 1 or dropoff_pct > 1:
        return _fail(
            f"More than 1% of Silver rows missing zone names: "
            f"pickup={pickup_pct:.2f}%, dropoff={dropoff_pct:.2f}%."
        )
    return _ok(f"Zone join: {pickup_pct:.2f}% null pickup, {dropoff_pct:.2f}% null dropoff (both <1%).")